from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import os
#os.chdir('/content/drive/MyDrive/name/IMP-OIC-Windowing')
from utils.extractframes import FrameExtractor
import graphene
directory_path = 'LifeQA/videos'
from gpt_ask import run_gpt

df1 = pd.read_json('LifeQA/questions_set/lqa_dev.json')
df2 = pd.read_json('LifeQA/questions_set/lqa_test.json')
df3 = pd.read_json('LifeQA/questions_set/lqa_train.json')

final_df = pd.concat([df1.T, df2.T,df3.T], ignore_index=True)
main = final_df.fillna('')
main

,automatic_captions,end_time,manual_captions,parent_video_id,questions,start_time
0,"[{'confidence': 0.8560221791267391, 'transcrip...",76.650667,"[{'transcript': '[boy] hello [girl] hey'}, {'t...",https://www.youtube.com/watch?v=TGUYv10XdTI,"[{'answer_type': 'V', 'answers': ['11', '4', '...",19.328
1,"[{'confidence': 0.9051294922828671, 'transcrip...",82.346667,[{'transcript': '[man] so hey crew we’re just ...,https://www.youtube.com/watch?v=W1_WorcmxPM,"[{'answer_type': 'L', 'answers': ['homework', ...",21.418667
2,"[{'confidence': 0.74139654636383, 'transcript'...",162.965333,[{'transcript': '[inaudible] [kid2] ok come on...,https://www.youtube.com/watch?v=W1_WorcmxPM,"[{'answer_type': 'B', 'answers': ['shiny lugga...",82.346667
3,"[{'confidence': 0.9240259528160091, 'transcrip...",254.762667,[{'transcript': '[kid2] [inaudible] there go m...,https://www.youtube.com/watch?v=W1_WorcmxPM,"[{'answer_type': 'V', 'answers': ['totally hap...",162.965333
4,"[{'confidence': 0.957232594490051, 'transcript...",320.405333,[{'transcript': '[kid1] and now [man] what are...,https://www.youtube.com/watch?v=W1_WorcmxPM,"[{'answer_type': 'V', 'answers': ['folding clo...",254.762667
...,...,...,...,...,...,...
270,"[{'confidence': 0.9013786911964411, 'transcrip...",126.592,[{'transcript': '[kid] wait am I gonna talk th...,https://www.youtube.com/watch?v=SrBp1Ojt5Z0,"[{'answer_type': 'L', 'answers': ['nickel', 'd...",17.962667
271,"[{'confidence': 0.915304899215698, 'transcript...",225.664,[{'transcript': '[woman] ok so James do you th...,https://www.youtube.com/watch?v=SrBp1Ojt5Z0,"[{'answer_type': 'B', 'answers': ['alcohol', '...",136.533333
272,"[{'confidence': 0.931367754936218, 'transcript...",55.123011,[{'transcript': '[woman] here we go Zoey we ha...,https://www.youtube.com/watch?v=R9lkOjLNzU4,"[{'answer_type': 'L', 'answers': ['Is more ora...",11.261678
273,"[{'confidence': 0.954777479171752, 'transcript...",174.401016,[{'transcript': '[woman] were gonna do one wit...,https://www.youtube.com/watch?v=R9lkOjLNzU4,"[{'answer_type': 'B', 'answers': ['sing', 'wal...",107.926349


In [2]:
#download the videos
from pytube import YouTube
path="LifeQA/videos"
#https://stackoverflow.com/questions/40713268/download-youtube-video-using-python-to-a-certain-directory
# videos unavailable or private
'''could not download https://www.youtube.com/watch?v=W1_WorcmxPM
could not download https://www.youtube.com/watch?v=2G5czNVhAJQ
could not download https://www.youtube.com/watch?v=0FgFZIEt5YQ
could not download https://www.youtube.com/watch?v=v0hJuq-WKnM
could not download https://www.youtube.com/watch?v=DjyFaMRTFj4
could not download https://www.youtube.com/watch?v=JS60qyA1Kws
could not download https://www.youtube.com/watch?v=8gamPx-DHAw
could not download https://www.youtube.com/watch?v=aaweXw03kQI
could not download https://www.youtube.com/watch?v=wM24NQYVZO8
could not download https://www.youtube.com/watch?v=R9lkOjLNzU4'''

def downloadYouTube(video_id):

    yt = YouTube(video_id)
    
    yt = yt.streams.filter(progressive=True, file_extension='mp4').order_by('resolution').desc().first()
    if not os.path.exists(path):
        os.makedirs(path)
    yt.download(path, filename = video_id.split('=')[1]+'.mp4')
    
v_ids = main['parent_video_id'].unique()
for v_id in v_ids:
    try:
        downloadYouTube(v_id)
    except Exception:
        print('could not download '+v_id)

could not download https://www.youtube.com/watch?v=TGUYv10XdTI
could not download https://www.youtube.com/watch?v=W1_WorcmxPM
could not download https://www.youtube.com/watch?v=2G5czNVhAJQ
could not download https://www.youtube.com/watch?v=h0tC1cGMMXw
could not download https://www.youtube.com/watch?v=D5dkBifv7K0


KeyboardInterrupt: 

In [20]:
diff =main.apply(lambda row: (int(row['end_time'])-int(row['start_time'])) if row['end_time'] and row['start_time'] else 0, axis=1).tolist()
diff

[57,
 61,
 80,
 92,
 66,
 62,
 65,
 67,
 66,
 63,
 63,
 0,
 0,
 0,
 0,
 0,
 63,
 62,
 75,
 55,
 70,
 84,
 107,
 73,
 96,
 70,
 68,
 65,
 65,
 68,
 60,
 60,
 95,
 89,
 104,
 86,
 93,
 62,
 72,
 87,
 67,
 0,
 65,
 120,
 60,
 69,
 66,
 65,
 72,
 63,
 61,
 77,
 63,
 62,
 68,
 64,
 65,
 66,
 63,
 76,
 98,
 76,
 110,
 64,
 67,
 71,
 82,
 72,
 94,
 112,
 79,
 79,
 94,
 100,
 69,
 65,
 103,
 67,
 76,
 68,
 119,
 79,
 63,
 60,
 70,
 80,
 88,
 61,
 62,
 62,
 80,
 72,
 80,
 86,
 58,
 63,
 0,
 64,
 59,
 78,
 73,
 63,
 60,
 61,
 72,
 59,
 63,
 59,
 63,
 67,
 65,
 64,
 65,
 71,
 121,
 56,
 66,
 61,
 42,
 60,
 64,
 104,
 61,
 101,
 67,
 117,
 49,
 80,
 116,
 61,
 62,
 61,
 71,
 77,
 60,
 0,
 0,
 0,
 0,
 0,
 0,
 92,
 61,
 77,
 114,
 69,
 0,
 81,
 57,
 67,
 88,
 67,
 67,
 68,
 65,
 46,
 73,
 69,
 73,
 67,
 65,
 74,
 118,
 65,
 63,
 67,
 71,
 59,
 73,
 58,
 72,
 69,
 63,
 86,
 61,
 60,
 65,
 81,
 75,
 75,
 87,
 62,
 111,
 83,
 43,
 67,
 73,
 94,
 88,
 79,
 98,
 65,
 97,
 70,
 91,
 61,
 70,
 75,
 121,
 7

In [2]:
import ffmpeg

def segment_video(input_path, output_path, start, end):
    input_file = ffmpeg.input(directory_path+'/'+input_path+'.mp4')
    duration = int(end-start)
    print(duration)
    #if duration < 10:
    output_file = ffmpeg.output(input_file.trim(start=start, duration = duration).filter('setpts','PTS-STARTPTS'), filename=directory_path+'/trim/'+output_path+'.mp4')
    ffmpeg.run(output_file)
    return True
    #else:
      #return False

In [3]:

def run_oic(question_id, video_id , start, end):
    out_dir = "out"
    # check if the directory has a video file
    for video in os.scandir(directory_path):
        video_name=(video.name).split('.')[0]
        if video_name == video_id:
            #segment the video into a clip with the given start and end timestamp
            video_p = segment_video(video_name,str(question_id)+video_name,start,end)
            # if video exists, extract frames with windowing of few frames to improve the detection
            if video_p:
                ex = FrameExtractor(video_file=directory_path+'/trim/'+str(question_id)+video_name+'.mp4',fps_to_save=10, window_size=2)
                ex.main()
                
                # instantiate graphene "OIC core that runs RelTR"  
                g = graphene.Graphene(alpha=0.3, min_assignment_conf=0.6)
            
                # check if the out directory exists else create one
                if not os.path.isdir(out_dir):
                    os.mkdir(out_dir)
                
                # prepare output files 
                text = str(question_id)+video_name+'graph2text.txt'
                img_dir_path = '{}/trim/{}{}-opencv'.format(directory_path,str(question_id),video_name)  
                
                # classify images from the image directory
                g.classify_images_window(img_dir_path,5)
                
                # generate relationship graph
                graph_dir_path =  "{}/img/JSON".format(img_dir_path)
                g.generate_temporal_graph_frames_no_plot(scenegraphs_path=graph_dir_path, image_path= "{}/img".format(img_dir_path))

                # save textual output in the out directory
                g.tg.to_text(os.path.join(out_dir, text))

                if os.path.isfile(os.path.join(out_dir,text)):
                    with open(os.path.join(out_dir,text)) as f:
                        context = "".join(map(str,f.readlines()))
                        return str(context)
                else:
                    continue

import shutil
shutil.rmtree('temp')

In [8]:
import math
print(main.shape[0])
main_liat=list(main['parent_video_id'])
not_found_vids = ['https://www.youtube.com/watch?v=W1_WorcmxPM',
'https://www.youtube.com/watch?v=2G5czNVhAJQ','https://www.youtube.com/watch?v=0FgFZIEt5YQ'
'https://www.youtube.com/watch?v=v0hJuq-WKnM',
'https://www.youtube.com/watch?v=JS60qyA1Kws',
'https://www.youtube.com/watch?v=8gamPx-DHAw',
'https://www.youtube.com/watch?v=aaweXw03kQI',
'https://www.youtube.com/watch?v=wM24NQYVZO8',
'https://www.youtube.com/watch?v=R9lkOjLNzU4']

remaining = [x for x in main_liat if x not in not_found_vids]

print(len(remaining))

v_ids = main['parent_video_id'].unique()
print(len(v_ids))
ds = pd.DataFrame()
for vid in v_ids:
    que = main.query("parent_video_id=='{}'".format(vid))
    video_id = str(que["parent_video_id"].values[0]).split('=')[1]
    questions = que['questions'].values[0]
    start = que['start_time'].values[0]
    end = que['end_time'].values[0]
    subtitles = que['manual_captions'].values[0]
    answer_type = que['answer_type'].values[0]
    subtitle = ''
    if len(subtitles)>1:
        for s in subtitles:
            subtitle += s['transcript'] + '\n'
    else: 
        subtitle = '_'
    for q in questions:
        question = q['question']
        options = q['answers']
        correct_ans_idx = q['correct_index']
        q_id = q['q_id']
        
        choice_string = ''

        choice_string = "0: {}, 1: {}, 2: {}, 3: {}".format(options[0], options[1],options[2],options[3])
        
        ds['q_id'] = q_id
        ds['question'] = question
        ds['answer_type'] = answer_type
        ds['Answers'] = options
        ds['ans_idx'] = correct_ans_idx
        ds['video_id'] = video_id
        ds['start'] = start
        ds['end'] = end
        ds['subtitles'] = subtitle
        
        
       

        if not os.path.isdir('{}/trim/{}{}-opencv'.format(directory_path,q_id,video_id)):
             prompt = run_oic(q_id, video_id, start, end)
             if prompt is not None:
                print(ds.info)
                ds['OIC_context'] = prompt
                formatted_question = question+ 'subtitle:'+str(subtitle) +' Guess the most likely answer among these options: '+choice_string+' Respond only with a single number between 0 and 4. Do not produce any other output.'
                response = run_gpt(prompt, formatted_question)
                ds['OIC_answer'] = str(response)
                ds['OIC_question'] = formatted_question
                count = 0
                while count<2:
                    if len(response)>1:
                        response = run_gpt(prompt, formatted_question) 
                        count+=1
                    else:
                        response = int(response)
                        break
                OIC_answer = response
                if not len(str(OIC_answer)) > 1:
                    if OIC_answer == correct_ans_idx:
                
                        ds['Match'] = 'Correct'
                        print('correct')
                    else:
                        print('wrong')

                        ds['Match'] = 'Wrong'
                else:
                    ds['Match'] = response
                
                print('-'*100)
                print('OIC question: {}'.format(formatted_question))
                print('OIC answer: {}'.format(OIC_answer))
                print('Answer: {}'.format(correct_ans_idx))
                ds.to_csv('LifeQA/datatables/question_answers.csv')

275
237
59


KeyError: 'answer_type'